In [66]:
import pandas as pd
import time
import datetime
import plotly.express as px
from pathlib import Path
import torch
from lib.models import MLP2hl
from lib.modules import evaluate_loop, plot_and_save_cm, summary, window_session, pad_for_windowing, predict_and_plot_pretty_session
from lib.utils import get_bouts_smoothed
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from datetime import timedelta
import numpy as np
import matplotlib.pyplot as plt

# CSV to PT

In [85]:
dates = []
lengths = []
times = []
times_from_n_samples = []
fps = []
raw_dir = Path('/home/musa/.delta/thrasher/')
for rec in raw_dir.iterdir():
    date = rec.name
    if (rec / 'acceleration.csv').exists():
        df = pd.read_csv(rec / 'acceleration.csv', skiprows=1)
        fps.append(str(rec / 'acceleration.csv'))
    else:
        df = pd.read_csv(rec / f'raw/{rec.name}.0.csv', skiprows=1, low_memory=False)
        df = df.rename(columns={'timestamp': 'timestamp', 'acc_x': 'x', 'acc_y': 'y', 'acc_z': 'z'})
        fps.append(str(rec / f'raw/{rec.name}.0.csv'))
    total_time = df['timestamp'].iloc[-1] - df['timestamp'].iloc[0]
    elapsed_time = timedelta(microseconds=int(total_time/1e3))
    time_from_n_samples = timedelta(seconds=len(df) / 100)
    dates.append(date)
    lengths.append(len(df))
    times.append(elapsed_time)
    times_from_n_samples.append(time_from_n_samples)

recs = pd.DataFrame({
    'date': dates,
    'length': lengths,
    'elapsed': times,
    'elapsed_from_n_samples': times_from_n_samples,
    'fp': fps
})
recs

,date,length,elapsed,elapsed_from_n_samples,fp
0,2022-12-20_11_18_54,141148,0 days 02:46:07.289585,0 days 00:23:31.480000,/home/musa/.delta/thrasher/2022-12-20_11_18_54...
1,2022-12-20_10_54_30,133303,0 days 02:15:45.205771,0 days 00:22:13.030000,/home/musa/.delta/thrasher/2022-12-20_10_54_30...
2,2022-12-06_17_27_56,66569,0 days 00:53:05.201043,0 days 00:11:05.690000,/home/musa/.delta/thrasher/2022-12-06_17_27_56...
3,2022-12-10_14_27_45,303421,0 days 04:02:24.940042,0 days 00:50:34.210000,/home/musa/.delta/thrasher/2022-12-10_14_27_45...
4,2022-12-12_07_03_40,369833,0 days 05:33:13.869892,0 days 01:01:38.330000,/home/musa/.delta/thrasher/2022-12-12_07_03_40...
5,2022-12-20_15_10_43,190558,0 days 02:47:20.921181,0 days 00:31:45.580000,/home/musa/.delta/thrasher/2022-12-20_15_10_43...
6,2022-12-21_14_16_02,110096,0 days 05:01:27.848092,0 days 00:18:20.960000,/home/musa/.delta/thrasher/2022-12-21_14_16_02...
7,2022-12-06_12_56_06,6616,0 days 00:21:12.890173,0 days 00:01:06.160000,/home/musa/.delta/thrasher/2022-12-06_12_56_06...
8,2022-12-19_08_37_36,327831,0 days 04:20:56.787042,0 days 00:54:38.310000,/home/musa/.delta/thrasher/2022-12-19_08_37_36...
9,2022-12-10_12_57_32,112618,0 days 01:30:00.776797,0 days 00:18:46.180000,/home/musa/.delta/thrasher/2022-12-10_12_57_32...


In [130]:
i = 17
print(recs.iloc[i])
rec = Path(recs.iloc[i]['fp'])
df = pd.read_csv(rec, skiprows=1, low_memory=False)
df = df.rename(columns={'timestamp': 'timestamp', 'acc_x': 'x', 'acc_y': 'y', 'acc_z': 'z'})
diffs = np.diff(df['timestamp'].to_numpy() / 1e9)
diffs = (diffs - diffs.min())
df['diffs (s)'] = np.concatenate([[0], diffs])
# fig = plt.plot(df['diffs (s)'])
# fig
fig = px.line(df[::1], y=['x', 'y', 'z'])
fig.show(renderer='browser')

date                                                    2022-12-06_15_48_49
length                                                                 2475
elapsed                                              0 days 00:01:56.912753
elapsed_from_n_samples                               0 days 00:00:24.750000
fp                        /home/musa/.delta/thrasher/2022-12-06_15_48_49...
Name: 17, dtype: object


In [131]:
date = rec.name.split('.')[0]
X = torch.Tensor(df[['x', 'y', 'z']].values)
print(X.shape)
torch.save(X, f'/home/musa/.delta/raw_pt/{date}.pt')

torch.Size([2475, 3])


# Score

In [3]:
data_dir = Path('../data/andrew/2023-10-26_15_32_20/')
acceleration = pd.read_csv(data_dir / 'acceleration.csv',skiprows=1).rename({'x':'x_acc', 'y':'y_acc', 'z':'z_acc'}, axis=1)
acceleration_start_time_seconds = float(pd.read_csv(data_dir / 'acceleration.csv',nrows=1,header=None).iloc[0,0].split()[-1])/1000
acceleration.timestamp = ((acceleration.timestamp - acceleration.timestamp[0])*1e-9)+acceleration_start_time_seconds
acceleration = acceleration.dropna()

start = int(datetime.datetime(2023, 10, 26, 16, 20, 0).strftime('%s'))
end = int(datetime.datetime(2023, 10, 26, 16, 37, 0).strftime('%s'))
acceleration['label'] = 0
acceleration.loc[(acceleration.timestamp > start) & (acceleration.timestamp < end),'label'] = 1

print(session_dir.name)
print(len(acceleration))
print(f'{timedelta(seconds=acceleration.timestamp.iloc[-1] - acceleration.timestamp.iloc[0])}')

FileNotFoundError: [Errno 2] No such file or directory: '/home/musa/.delta/acceleration.csv'

In [4]:
Xte = torch.Tensor(acceleration[['x_acc','y_acc','z_acc']].values)
yte = torch.Tensor(acceleration['label'].values).unsqueeze(1)

Xte = pad_for_windowing(Xte, WINSIZE)
Xte = window_session(Xte, WINSIZE)

In [5]:
model = MLP2hl([20,20], WINSIZE).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters())
criterion = nn.BCEWithLogitsLoss()

In [12]:
model.load_state_dict(torch.load(Path('dev/mlp2hl/best_model.pt')))
# model.load_state_dict(torch.load(Path('dev/mlp2hl_3/model/1.pt')))

<All keys matched successfully>

In [13]:
testloader = DataLoader(TensorDataset(Xte,yte), batch_size=64)
ys,metrics = evaluate_loop(model, criterion, testloader, DEVICE)
plot_and_save_cm(ys['true'], ys['pred'])
summary(metrics)

KeyboardInterrupt: 

In [ ]:
# Cut off part of session
buffer = 700_000
dim_factor = 10

first_pos = max(acceleration.loc[acceleration['label']==1].index[0]-buffer, 0)
last_pos = min(acceleration.loc[acceleration['label']==1].index[-1]+buffer, len(acceleration))

session = acceleration.loc[first_pos:last_pos-1, ['x_acc','y_acc','z_acc', 'label']].reset_index(drop=True)
session['Predicted Eating'] = ys['pred'][first_pos:last_pos]
session['Confidence'] = ys['conf'][first_pos:last_pos]

pred_bouts = get_bouts_smoothed(ys['pred'][first_pos:last_pos])

fig = px.line(
    session[::dim_factor], 
    x=session.index[::dim_factor], 
    y=['x_acc','y_acc','z_acc', 'Predicted Eating', 'Confidence']
)
fig.add_vrect(
    x0=session.loc[session['label']==1].index[0], 
    x1=session.loc[session['label']==1].index[-1], 
    fillcolor='black', 
    opacity=.2,
    line_width=0,
    layer="below"
)
for bout in pred_bouts:
    fig.add_vrect(
        x0=bout['start'], 
        x1=bout['end'], 
        fillcolor='red', 
        opacity=.2,
        line_width=0,
        layer="below"
    )
fig.show(renderer='browser')

In [ ]:
fig.write_html('dev/mlp2hl_andrew_plot.html')